In [1]:
# Install the tsfm library
! pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.2.22"
# Install a utility to help download data files from google drive during the data prep process
! pip install gdown

  Cloning https://github.com/ibm-granite/granite-tsfm.git (to revision v0.2.22) to /private/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/pip-install-fny9wfq3/granite-tsfm_1e5fc1702bcd47349d03b434049f288c
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite/granite-tsfm.git /private/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/pip-install-fny9wfq3/granite-tsfm_1e5fc1702bcd47349d03b434049f288c
  Running command git checkout -q 216850d0cb073e31689049c1334f701fe11bc2c3
  Resolved https://github.com/ibm-granite/granite-tsfm.git to commit 216850d0cb073e31689049c1334f701fe11bc2c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for granite-tsfm: filename=granite_tsfm-0.2.22-py3-none-any.whl size=2341459 sha256=200ada64312f3a0b13a461d98e335798a606a7655fc0232662d3daa545072247
  Stored in directory: /private/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/p

In [ ]:
import math
import os

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Subset
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed

from tsfm_public import (
    ForecastDFDataset,
    TimeSeriesForecastingPipeline,
    TimeSeriesPreprocessor,
    TinyTimeMixerForPrediction,
    TrackingCallback,
    count_parameters,
)
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.time_series_preprocessor import prepare_data_splits
from tsfm_public.toolkit.visualization import plot_predictions
import evaluate

## Load data from carOBD

In [83]:
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata" 
time_col = 'ENGINE_RUN_TINE ()'

df_list = []
for file in os.listdir(path):
  if file.endswith('.csv'):
    df = pd.read_csv(f'{path}/{file}', index_col=False)
    df['filename'] = file
    df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')


129 files loaded out of 129


## Remove zero-variance columns

In [84]:
def remove_zero_variance_columns(df: pd.DataFrame) -> pd.DataFrame:
  """
  Compute std of each std-computable column (numeric only)
  """
  std_df = df.std(numeric_only=True)

  zero_variance_cols = std_df[std_df == 0].index.tolist()
  print(f'{len(zero_variance_cols)} columns with zero variance')

  if len(zero_variance_cols) > 0:
    df = df.drop(columns=zero_variance_cols)

  return df

## Handle missing Timestamps and duplicates

In [85]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
  """
  Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
  This preserves the overall statistics while removing duplicate entries.
  
  Note: The time column itself is not averaged (it becomes the group key).
  Only numeric columns are averaged when multiple rows share the same timestamp.
  """
  df = df.groupby(time_col, as_index=False).mean(numeric_only=True)
  return df

## Per-channel Standardisation

In [86]:
from sklearn.preprocessing import StandardScaler

def normalise_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
  """
  Normalise numeric columns without Timestamp
  """
  scaler = StandardScaler()
  normalised_array = scaler.fit_transform(df.select_dtypes(include=['number']).drop(columns=[time_col]))
  normalised_df = pd.DataFrame(normalised_array, columns=df.select_dtypes(include=['number']).columns.drop(time_col))
  normalised_df.insert(0, time_col, df[time_col])
  return normalised_df

## Create sliding windows

In [91]:
# we don't want any leakage between files so we treat each file as a separate time series
def create_sliding_windows(df: pd.DataFrame, window_size: int = 512,  step_size: int=1,) -> pd.DataFrame:
  """
  Create an array of window_size arrays. Option to downsample the data. 1: 1 sample per second, 2: 1 sample every 2 seconds, etc
  By default, window_size is 512 per TSPulse-R2 docs
  """
  windowed_list = []
  for i in range(0, len(df), step_size):
    window = df.iloc[i:i+window_size]
    windowed_list.append(window)
  return windowed_list

## Processed Dataframe

In [92]:
processed_df = df_list[0]
processed_df = mean_fill_missing_timestamps_and_remove_duplicates(processed_df)
processed_df = remove_zero_variance_columns(processed_df)
processed_df = normalise_numeric_columns(processed_df)
processed_df = create_sliding_windows(processed_df, 1)


[df.shape for df in processed_df]

5 columns with zero variance


[(1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),
 (1, 22),


## Inject Anomalies

## Metrics Function